In [2]:
import pandas as pd
import csv
import re

### MultiPrompts with one Agent ensemble ###

In [3]:
multi_hop = pd.read_csv('/home/smjo/KG-gpt2/ver_8_multiPrompts/results_test/gpt-4o-mini_maxiter_15/multi_hop/only_result.csv', header=None)
multi_claim = pd.read_csv('/home/smjo/KG-gpt2/ver_8_multiPrompts/results_test/gpt-4o-mini_maxiter_15/multi_claim/only_result.csv', header=None)
num1 = pd.read_csv('/home/smjo/KG-gpt2/ver_8_multiPrompts/results_test/gpt-4o-mini_maxiter_15/num1/only_result.csv', header=None)
exist = pd.read_csv('/home/smjo/KG-gpt2/ver_8_multiPrompts/results_test/gpt-4o-mini_maxiter_15/existence/only_result.csv', header=None)

In [4]:
len(multi_claim), len(multi_hop), len(num1), len(exist)

(100, 100, 100, 100)

In [5]:
def preprocess(answer):
    answer = re.sub(r"[^a-zA-Z0-9]", "",answer.lower())
    return answer

In [10]:
correct, wrong, abstain=0,0,0
main_only= exist
for idx in range(len(main_only)):
    qid = main_only.loc[idx,0]
    pred_list = main_only.loc[idx,1].strip().split(',')
    label = preprocess(str(main_only.loc[idx,2]))
    
    
    pred1, pred2, pred3= preprocess(pred_list[0]), preprocess(pred_list[1]), preprocess(pred_list[2])
    new_pred_list = [pred1, pred2, pred3]
    
    

    if 'abstain' in new_pred_list:
        abstain +=1
        continue
    if pred1==pred2==pred3 : 
        final_answer = pred1
        if final_answer == label:
            correct+=1
        else:
            wrong+=1
        continue
    else:
        abstain +=1
    

mtr1 = (len(main_only)-abstain)/ len(main_only)
mtr2 = correct/ (len(main_only)-abstain)
mtr3 = (correct-wrong)/ (len(main_only)-abstain)


print(mtr1, mtr2, mtr3)

0.53 0.9622641509433962 0.9245283018867925


In [4]:
import pandas as pd
from sklearn.metrics import f1_score

# Load the data from a CSV file without headers
file_path = '/home/smjo/KG-gpt2/ver_8_multiPrompts/results_test/gpt-3.5/maxiter_15/multi_claim/only_result.csv'  # Replace with your actual file path
df = pd.read_csv(file_path, header=None)

# Assign column names manually
df.columns = ['qid', 'prediction', 'gt_label']

# Convert prediction strings to lists
def parse_prediction(prediction):
    return eval(prediction)  # Safely convert string representation of list to actual list

df['prediction'] = df['prediction'].apply(parse_prediction)

# Determine the final prediction
def final_prediction(predictions):
    if predictions.count(predictions[0]) == 3:
        return predictions[0]  # All predictions are the same
    elif predictions.count(predictions[0]) > 1:
        return predictions[0]  # Majority prediction
    else:
        return 'Abstain'  # No common agreement

# Apply the final_prediction function
df['final_prediction'] = df['prediction'].apply(lambda x: final_prediction(x))

# Calculate Metric 1
metric_1 = len(df[df['final_prediction'] != 'Abstain']) / len(df)

# Filter out rows where the final prediction is 'Abstain'
filtered_df = df[df['final_prediction'] != 'Abstain']

# Calculate the F1 score
y_true = filtered_df['gt_label']
y_pred = filtered_df['final_prediction'].apply(lambda x: x == 'True')

micro_f1 = f1_score(y_true, y_pred)

# Display results
print(f"Metric 1: {metric_1:.4f}")
print(f"Micro-F1 Score: {micro_f1:.4f}")


Metric 1: 0.2900
Micro-F1 Score: 0.8889


In [12]:
import csv

def calculate_metrics(file_path):
    # Initialize variables
    f1_scores = []
    total_samples = 0
    abstain_samples = 0
    total_tp = total_fp = total_fn = 0  # For Macro F1 Score
    hit_count = 0  # For Hit@1
    valid_samples = 0  # For final prediction that is not abstain

    # Open and read the CSV file
    with open(file_path, 'r') as file:
        reader = csv.reader(file)
        next(reader)  # Skip header row
        for row in reader:
            total_samples += 1  # Count total samples

            # Parse data
            sample_id = row[0]
            predictions = eval(row[1])  # Safely parse the list of predictions
            print(predictions)
            ground_truth = row[2].strip().lower() == 'true'  # Convert ground truth to boolean

            # Determine if final prediction is valid
            if 'abstain' in [p.lower() for p in predictions] or len(set(predictions)) > 1:
                abstain_samples += 1
                continue

            valid_samples += 1

            # Determine final prediction
            final_prediction = predictions[0].strip().lower() == 'true'
            print(final_prediction)

            # Calculate TP, FP, FN for this sample
            tp = 1 if final_prediction and ground_truth else 0
            fp = 1 if final_prediction and not ground_truth else 0
            fn = 1 if not final_prediction and ground_truth else 0

            total_tp += tp
            total_fp += fp
            total_fn += fn

            # Calculate precision, recall, and F1 score for this sample
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0

            f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            f1_scores.append(f1_score)

            # Calculate Hit@1
            if final_prediction == ground_truth:
                hit_count += 1

    # Calculate Coverage
    coverage = (total_samples - abstain_samples) / total_samples if total_samples > 0 else 0

    # Calculate Macro F1 Score
    macro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    macro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    macro_f1_score = (2 * macro_precision * macro_recall) / (macro_precision + macro_recall) if (macro_precision + macro_recall) > 0 else 0

    # Calculate Sample-wise F1 Score
    sample_wise_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0

    # Calculate Hit@1
    hit_at_1 = hit_count / valid_samples if valid_samples > 0 else 0

    return coverage, macro_f1_score, sample_wise_f1, hit_at_1

# Example usage
file_path = '/home/smjo/KG-gpt2/ver_8_multiPrompts/results_test/gpt-4o-mini_maxiter_15/multi_hop/only_result.csv'  # Replace with your CSV file path
coverage, macro_f1, sample_wise_f1, hit_at_1 = calculate_metrics(file_path)
print(f"Coverage: {coverage:.4f}")
print(f"Macro F1 Score: {macro_f1:.4f}")
print(f"Sample-wise F1 Score: {sample_wise_f1:.4f}")
print(f"Hit@1: {hit_at_1:.4f}")


['True', 'True', 'Abstain']
['True', 'True', 'Abstain']
['False', 'Abstain', 'False']
['True', 'Abstain', 'Abstain']
['True', 'True', 'Abstain']
['True', 'True', 'Abstain']
['True', 'True', 'True']
True
['False', 'False', 'Abstain']
['False', 'False', 'True']
['True', 'True', 'True']
True
['False', 'Abstain', 'False']
['Abstain', 'True', 'Abstain']
['True', 'False', 'True']
['Abstain', 'Abstain', 'Abstain']
['Abstain', 'False', 'Abstain']
['True', 'True', 'Abstain']
['True', 'True', 'False']
['True', 'True', 'Abstain']
['True', 'True', 'True']
True
['True', 'True', 'True']
True
['Abstain', 'Abstain', 'False']
['False', 'Abstain', 'False']
['True', 'False', 'False']
['False', 'False', 'False']
False
['True', 'True', 'Abstain']
['False', 'False', 'False']
False
['False', 'True', 'False']
['True', 'True', 'True']
True
['True', 'Abstain', 'Abstain']
['False', 'False', 'False']
False
['True', 'Abstain', 'False']
['False', 'False', 'Abstain']
['False', 'False', 'False']
False
['True', 'True'

In [16]:
import pandas as pd

# 데이터 로드 (csv 파일에서 DataFrame으로 읽기)
file_path = '/home/smjo/KG-gpt2/ver_7_factKG/results_test/gpt-4o-mini/15_only_result_multi_hop.csv'  # CSV 파일 경로
df = pd.read_csv(file_path)

# Abstain을 제외한 데이터 필터링
filtered_df = df[df["prediction"].str.strip().str.lower() != "abstain"]
print("Filtered DataFrame Size:", len(filtered_df))  # 필터링된 데이터 크기 확인

# 예측값과 실제값을 True/False로 변환 (문자열이 아닌 경우를 대비해 str 처리 추가)
predictions = filtered_df["prediction"].astype(str).str.strip().str.lower().map(lambda x: x == "true").tolist()
ground_truths = filtered_df["gt_label"].astype(str).str.strip().str.lower().map(lambda x: x == "true").tolist()

# Micro F1 Score 계산 (직접 구현)
TP = sum((p == True and g == True) for p, g in zip(predictions, ground_truths))
FP = sum((p == True and g == False) for p, g in zip(predictions, ground_truths))
FN = sum((p == False and g == True) for p, g in zip(predictions, ground_truths))

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
micro_f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# Sample-wise F1 Score 계산
sample_f1_scores = []
for pred, gt in zip(predictions, ground_truths):
    if pred == gt == True:
        sample_f1_scores.append(1.0)  # TP
    elif pred == gt == False:
        sample_f1_scores.append(1.0)  # TN
    elif pred != gt:
        sample_f1_scores.append(0.0)  # FN or FP
sample_wise_f1 = sum(sample_f1_scores) / len(sample_f1_scores)

# Hit@1 계산
hits = sum(p == g for p, g in zip(predictions, ground_truths))
hit_at_1 = hits / len(predictions)

# Coverage 계산
total_samples = len(df)
abstain_samples = len(df[df["prediction"].str.strip().str.lower() == "abstain"])
coverage = (total_samples - abstain_samples) / total_samples

# 결과 출력
print("Micro F1 Score:", micro_f1)
print("Sample-wise F1 Score:", sample_wise_f1)
print("Hit@1:", hit_at_1)
print("Coverage:", coverage)


Filtered DataFrame Size: 43
Micro F1 Score: 0.8571428571428572
Sample-wise F1 Score: 0.8372093023255814
Hit@1: 0.8372093023255814
Coverage: 0.43
